## Multithreading With Thread Pool Executor

#### Multithreading with ThreadPoolExecutor is a high-level, efficient way to manage multiple threads in Python using the concurrent.futures module.

Instead of manually creating, starting, and joining individual threads (which is tedious and resource-heavy), you create a "pool" of worker threads that stay alive and "check out" tasks from a queue as they become available.

In [1]:
from concurrent.futures import ThreadPoolExecutor
import time

def print_number(number):
    time.sleep(1)
    return f"Number :{number}"

numbers=[1,2,3,4,5]

with ThreadPoolExecutor(max_workers=3) as executor:
    results = executor.map(print_number,numbers)

for result in results:
    print(result)

Number :1
Number :2
Number :3
Number :4
Number :5


This code is more efficient because it switches from **sequential execution** (one-by-one) to **concurrent execution** (multiple at once).

### 1. Why it is more efficient
Imagine you are at a grocery store with 5 items to scan. 

* **Sequential (Normal for-loop):** There is only one cashier. They scan item 1, then item 2, and so on. If each item takes 1 second, you're there for **5 seconds**.
* **ThreadPoolExecutor (Your code):** The store opens **3 checkout lanes** (`max_workers=3`). 
    * The first 3 items are scanned simultaneously in different lanes. 
    * At the 1-second mark, those 3 are done.
    * The remaining 2 items then jump into the now-empty lanes.
    * At the 2-second mark, everything is finished.



**The Math:**
* **Sequential time:** $5 \text{ tasks} \times 1 \text{ second} = 5 \text{ seconds}$.
* **Your code's time:** $\approx 2 \text{ seconds}$ (because it processes 3, then 2).

---

### 2. Breaking down the Syntax
That specific block of code is doing a lot of heavy lifting behind the scenes. Here is the breakdown:

#### `with ThreadPoolExecutor(max_workers=3) as executor:`
This is a **Context Manager**. 
* **`ThreadPoolExecutor(max_workers=3)`:** This initializes the "pool" and tells Python, "I want a maximum of 3 threads working at any given time."
* **`with ... as executor`:** This is the "safety net." It ensures that as soon as the code inside the block is finished, all the threads are cleaned up and shut down properly. Without the `with` statement, you’d have to manually call `executor.shutdown()`.

#### `results = executor.map(print_number, numbers)`
This is a high-level way to distribute work. 
* **`executor.map`:** This works exactly like Python’s built-in `map(function, list)`. It takes every item in `numbers` and feeds it into `print_number`.
* **The Magic:** Unlike a regular map, it doesn't wait for the first number to finish before starting the second. It hands the numbers out to the 3 available workers as fast as possible.
* **The Order:** `executor.map` is special because it returns the results **in the order they were sent**, even if a later task finishes faster than an earlier one.

---

### When would this *not* be efficient?
If you remove the `time.sleep(1)` and replace it with a massive math calculation (like `number * 9999999`), you might find that this code doesn't speed up much in standard Python. 

This is because of the **Global Interpreter Lock (GIL)**. Standard Python threads are great at "waiting" (I/O bound tasks like downloading files or sleeping), but they struggle to do heavy "thinking" (CPU-bound tasks) at the same time. For heavy math, you'd switch `ThreadPoolExecutor` to `ProcessPoolExecutor`.



Since you're using `time.sleep`, you've picked the perfect scenario for multithreading. Are you planning to use this for something like web scraping or API calls?

## Multiprocessing With Process Pool Executor

#### Multiprocessing with ProcessPoolExecutor is the high-level way to run CPU-heavy tasks in parallel by bypassing Python’s Global Interpreter Lock (GIL).

Instead of multiple threads sharing one CPU core, you create a "pool" of entirely separate Python processes, each with its own memory space and its own CPU core. This allows you to crunch massive amounts of data or perform complex math across your entire processor, literally doing multiple things at the exact same instant.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import time

def square_number(number):
    time.sleep(1)
    return f"Square: {number*number}"

numbers=[1,2,3,4,5]

with ProcessPoolExecutor(max_workers=3) as executor:
    results = executor.map(square_number, numbers)

for result in results:
    print(result)

In [4]:
## Result(Executed in seperate file):
# Desktop\Cloud AI Backend\AI ML DataScience Practice\AI-ML-DataScience-Practice> py '.\Day 14\multiprocessingUsingProcessPoolExecutor.py'
# Square: 1
# Square: 4
# Square: 9
# Square: 16
# Square: 25